In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('shop_analysis.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product TEXT,
    amount REAL,
    order_date TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
)
''')

cursor.executemany('INSERT OR IGNORE INTO customers VALUES (?, ?, ?)', [
    (1, 'Anna', 'Warsaw'),
    (2, 'Jan', 'Krakow'),
    (3, 'Katarzyna', 'Warsaw'),
    (4, 'Piotr', 'Wroclaw')
])

cursor.executemany('INSERT OR IGNORE INTO orders VALUES (?, ?, ?, ?, ?)', [
    (101, 1, 'Laptop', 3500.0, '2026-01-10'),
    (102, 2, 'Phone', 2000.0, '2026-01-12'),
    (103, 1, 'Mouse', 150.0, '2026-01-15'),
    (104, 3, 'Monitor', 1200.0, '2026-01-18'),
    (105, 4, 'Keyboard', 300.0, '2026-01-20'),
    (106, 2, 'Monitor', 1200.0, '2026-01-22')
])
conn.commit()

query_1 = '''
SELECT c.name, c.city, SUM(o.amount) as total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC;
'''

df_result1 = pd.read_sql_query(query_1, conn)
display(df_result1)

query_2 = '''
SELECT c.city, c.name, o.product, o.amount,
       ROW_NUMBER() OVER(PARTITION BY c.city ORDER BY o.amount DESC) as rank_in_city
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id;
'''

df_result2 = pd.read_sql_query(query_2, conn)
display(df_result2)

conn.close()

,name,city,total_spent
0,Anna,Warsaw,3650.0
1,Jan,Krakow,3200.0
2,Katarzyna,Warsaw,1200.0
3,Piotr,Wroclaw,300.0


,city,name,product,amount,rank_in_city
0,Krakow,Jan,Phone,2000.0,1
1,Krakow,Jan,Monitor,1200.0,2
2,Warsaw,Anna,Laptop,3500.0,1
3,Warsaw,Katarzyna,Monitor,1200.0,2
4,Warsaw,Anna,Mouse,150.0,3
5,Wroclaw,Piotr,Keyboard,300.0,1
